# 02. Comprensión de los datos

Se describe el corpus integrado sin modificarlo. Se revisan fuente, textos vacíos, repeticiones y longitud. Los resultados agregados se guardan en JSON; no se crean columnas permanentes.


La reproducción lee las exportaciones guardadas y escribe en `salidas/<ejecución>/TABLAS`.

Si abres este cuaderno directamente en Colab, ejecuta su primera celda y autoriza Google Drive cuando se solicite. Para recorrer toda la secuencia de una vez, usa `INICIAR_EN_COLAB.ipynb`.

**Salidas guardadas.** Las tablas, figuras y registros visibles proceden de la corrida original `corrida_03_epocas`. Esta copia incorpora las preguntas de carga de 06–08. Las salidas y los números de ejecución conservan su origen histórico. Abre el cuaderno y desplázate para exponer los resultados.

La copia pública presenta el código de esta etapa. Sus salidas por publicación se conservan en la copia de Drive.

## A. Cargar la integración

- **Qué hace:** abre la salida del cuaderno 01.
- **Para qué:** trabajar con la misma versión del corpus.
- **Por qué:** una etapa debe consumir directamente la salida de la etapa anterior.


In [ ]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)

from rutas import carpeta_salidas

from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "corrida.json").is_file():
    raise FileNotFoundError("Abre Jupyter en la carpeta que contiene corrida.json.")
print("Carpeta de trabajo:", RAIZ)

import json
import re
import pandas as pd
from IPython.display import display

ENTRADA = carpeta_salidas(RAIZ) / "TABLAS/01_integracion/datos_integrados_v4.xlsx"
datos = pd.read_excel(ENTRADA, sheet_name="datos", keep_default_na=False)

if list(datos.columns) != ["id", "fuente", "texto_real"]:
    raise ValueError("La entrada no tiene el esquema mínimo del cuaderno 01.")
if not datos["id"].is_unique:
    raise ValueError("Hay identificadores repetidos.")

print("Filas:", len(datos))
display(datos.head(10))


## B. Revisar cantidad y calidad básica

- **Qué hace:** cuenta publicaciones por fuente, textos vacíos, repeticiones exactas y longitudes.
- **Para qué:** conocer problemas antes de limpiar o modelar.
- **Por qué:** las decisiones de preparación deben partir de evidencia del corpus, no de supuestos.


In [ ]:
texto = datos["texto_real"].astype(str)
no_vacio = texto.str.strip().ne("")
palabras = texto.str.findall(r"\S+").str.len()

resumen_fuente = datos.groupby("fuente").size().rename("filas").reset_index()
resumen_calidad = pd.DataFrame([{
    "filas": int(len(datos)),
    "textos_vacios": int((~no_vacio).sum()),
    "repeticiones_exactas_adicionales": int(texto[no_vacio].duplicated().sum()),
    "palabras_mediana": float(palabras[no_vacio].median()),
    "palabras_maximo": int(palabras[no_vacio].max()),
}])

display(resumen_fuente)
display(resumen_calidad)


## C. Visualizar la composición y longitud

- **Qué hace:** genera tres gráficos: publicaciones por fuente, porcentaje de textos vacíos y distribución de palabras.
- **Para qué:** reconocer rápidamente el equilibrio entre fuentes y los problemas que afectarán al modelado.
- **Por qué:** una tabla entrega cifras exactas, mientras que el gráfico permite comparar magnitudes y formas de la distribución.

El gráfico de longitud muestra hasta el percentil 95 para que los textos extremadamente largos no compriman toda la figura. Esos textos **no se eliminan** del dataset. Los gráficos describen únicamente este corpus y no prueban representatividad de la población boliviana.


In [ ]:
from rutas import carpeta_salidas

import matplotlib.pyplot as plt

CARPETA_GRAFICOS = carpeta_salidas(RAIZ) / "TABLAS/02_exploracion"
CARPETA_GRAFICOS.mkdir(parents=True, exist_ok=True)
colores = {"telegram": "#2A6FBB", "facebook": "#E07A2D"}

# 1. Cantidad de publicaciones por fuente
conteos = datos["fuente"].value_counts().reindex(["telegram", "facebook"])
fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(conteos.index, conteos.values, color=[colores[x] for x in conteos.index])
ax.set_title("Publicaciones recopiladas por fuente")
ax.set_xlabel("Fuente")
ax.set_ylabel("Número de publicaciones")
ax.bar_label(barras, padding=3)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(CARPETA_GRAFICOS / "grafico_fuentes_v4.png", dpi=160, bbox_inches="tight")
plt.show()

# 2. Porcentaje de textos vacíos por fuente
calidad = datos.assign(texto_vacio=~no_vacio).groupby("fuente")["texto_vacio"].agg(["sum", "count"])
calidad = calidad.reindex(["telegram", "facebook"])
porcentaje_vacio = 100 * calidad["sum"] / calidad["count"]
fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(porcentaje_vacio.index, porcentaje_vacio.values,
                color=[colores[x] for x in porcentaje_vacio.index])
ax.set_title("Textos vacíos por fuente")
ax.set_xlabel("Fuente")
ax.set_ylabel("Porcentaje de publicaciones (%)")
ax.bar_label(barras, labels=[f"{v:.2f}%" for v in porcentaje_vacio], padding=3)
ax.set_ylim(0, max(1, porcentaje_vacio.max() * 1.25))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(CARPETA_GRAFICOS / "grafico_textos_vacios_v4.png", dpi=160, bbox_inches="tight")
plt.show()




## D. Guardar el resumen

- **Qué hace:** exporta únicamente los conteos agregados.
- **Para qué:** documentar la comprensión de datos en la tesis.
- **Por qué:** estas medidas describen el corpus, pero no son variables de entrada para BERT.


In [ ]:
from rutas import carpeta_salidas

informe = {
    "version": "v4",
    "entrada": "TABLAS/01_integracion/datos_integrados_v4.xlsx",
    "filas": int(len(datos)),
    "por_fuente": {str(k): int(v) for k, v in datos["fuente"].value_counts().items()},
    "textos_vacios": int((~no_vacio).sum()),
    "repeticiones_exactas_adicionales": int(texto[no_vacio].duplicated().sum()),
    "palabras_mediana": float(palabras[no_vacio].median()),
    "palabras_maximo": int(palabras[no_vacio].max()),
    "graficos": [
        "grafico_fuentes_v4.png",
        "grafico_textos_vacios_v4.png",
        "grafico_longitud_v4.png",
    ],
    "filas_eliminadas": 0,
}
SALIDA = carpeta_salidas(RAIZ) / "TABLAS/02_exploracion/resumen_exploracion_v4.json"
SALIDA.parent.mkdir(parents=True, exist_ok=True)
SALIDA.write_text(json.dumps(informe, ensure_ascii=False, indent=2), encoding="utf-8")
print("Resumen guardado:", SALIDA)
